# Climatic attributes

Notebook to create the file `CAMELS_DE_1h_climatic_attributes.csv`.  

columns:
- gauge_id
- p_mean [mm/hour]
- p_seasonality [-]
- frac_snow [-]
- high_prec_freq [hours/yr]
- high_prec_dur [hours]
- high_prec_timing [season]
- low_prec_freq [hours/yr]
- low_prec_dur [hours]
- low_prec_timing [season]

In [1]:
from glob import glob
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from tqdm import tqdm

In [2]:
ids = [f.split("_")[-1].split(".")[0] for f in glob("../CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_*.csv")]

print(f"Total number of stations in CAMELS-DE-1h: {len(ids)}")

Total number of stations in CAMELS-DE-1h: 1611


CAMELS-CH: [Note that observed discharge is the most limiting variable. Hence, if discharge was only available for three hydrologic years for a certain basin to allow for the calculation of hydrologic signatures, the climatic indices were evaluated only for the same 3 years for the sake of consistency.](https://essd.copernicus.org/articles/15/5755/2023/#:~:text=.%20Note%20that%20observed%20discharge%20is%20the%20most%20limiting%20variable.)  

We do the same for CAMELS-DE.


In [3]:
import polars as pl
from datetime import datetime

def filter_complete_hydro_years(df: pl.DataFrame, tolerance: float = 0.05) -> pl.DataFrame:
    """
    Helper function to filter a DataFrame to only include complete hydrological 
    years (October - September). A hydrological year is considered complete if 
    it has less than the specified tolerance of missing values.
    """
    # Ensure time column is datetime
    if "time" not in df.columns:
        raise ValueError("DataFrame must have a 'time' column")
    
    # Parse time column if it's a string (handle timezone if present)
    if df["time"].dtype == pl.Utf8:
        df = df.with_columns([
            pl.col("time").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S%#z").alias("time")
        ])
    
    # Remove timezone if present (convert to timezone-naive) - BEFORE getting min/max
    if df["time"].dtype.time_zone is not None:
        df = df.with_columns([
            pl.col("time").dt.replace_time_zone(None).alias("time")
        ])
    
    # Cast to microsecond precision to ensure join works
    df = df.with_columns([
        pl.col("time").cast(pl.Datetime("us")).alias("time")
    ])
    
    # Get min and max years
    min_year = df["time"].dt.year().min()
    max_year = df["time"].dt.year().max()
    
    # Create start and end dates for complete hydrological years
    start_date = datetime(min_year - 1, 10, 1, 0, 0, 0)
    end_date = datetime(max_year + 1, 9, 30, 23, 0, 0)
    
    # Create complete hourly date range with microsecond precision
    complete_range = pl.datetime_range(
        start_date, 
        end_date, 
        interval="1h",
        time_unit="us",
        eager=True
    ).alias("time")
    
    # Create DataFrame with complete range and join with existing data
    df_complete = pl.DataFrame({"time": complete_range})
    df = df_complete.join(df, on="time", how="left")
    
    # Calculate hydrological year
    df = df.with_columns([
        pl.when(pl.col("time").dt.month() >= 10)
        .then(pl.col("time").dt.year() + 1)
        .otherwise(pl.col("time").dt.year())
        .alias("hydro_year")
    ])
    
    # Calculate missing value percentage per hydrological year
    missing_per_year = (
        df.group_by("hydro_year")
        .agg([
            pl.col("discharge_vol_obs").is_null().mean().alias("missing_pct")
        ])
    )
    
    # Get years that meet the tolerance threshold
    valid_years = missing_per_year.filter(
        pl.col("missing_pct") <= tolerance
    )["hydro_year"]
    
    # Filter DataFrame to only include valid years
    df_filtered = df.filter(pl.col("hydro_year").is_in(valid_years))
    
    # Drop the hydro_year column
    df_filtered = df_filtered.drop("hydro_year")
    
    return df_filtered

In [4]:
# dataframe to store results
df_results = pd.DataFrame()

## Function Definitions

Functions to calculate each climatic attribute from the processed dataframe.

In [10]:
def calculate_p_mean(df: pl.DataFrame) -> float:
    """
    Calculate mean hourly precipitation [mm/hour]
    """
    if df.height == 0:
        return np.nan
    
    p_mean = df["precipitation_mean"].mean()
    
    if p_mean is None:
        return np.nan
    
    return round(p_mean, 2)


def calculate_precipitation_perc_complete(df: pl.DataFrame) -> float:
    """
    Calculate percentage of non-missing precipitation values in the timeseries [%]
    """
    if df.height == 0:
        return np.nan
    
    n_total = df.height
    n_valid = df["precipitation_mean"].is_not_null().sum()
    
    return round(n_valid / n_total * 100, 2)


def calculate_p_seasonality(df: pl.DataFrame) -> float:
    """
    Calculate seasonality and timing of precipitation using sine curves.
    Positive values indicate precipitation peaks in summer, negative in winter.
    Works with hourly data by aggregating to daily for seasonality analysis.
    """
    if df.height == 0:
        return np.nan
    
    # Aggregate to daily for seasonality calculation (annual patterns are daily-scale)
    df_daily = df.group_by(pl.col("time").dt.date()).agg([
        pl.col("precipitation_mean").sum().alias("precipitation_mean"),
        pl.col("air_temperature_mean").mean().alias("air_temperature_mean")
    ]).sort(pl.col("time"))
    
    # Add date as datetime
    df_daily = df_daily.with_columns([
        pl.col("time").cast(pl.Datetime("us"))
    ])
    
    # Define the sine function to fit
    def sine_curve(day_of_year, mean_value, amplitude, phase_shift):
        return mean_value * (1 + amplitude * np.sin(2 * np.pi * (day_of_year - phase_shift) / 365.25))
    
    # Create a day_of_year column
    df_daily = df_daily.with_columns([
        pl.col("time").dt.ordinal_day().alias("day_of_year")
    ])

    # Get the mean precipitation and temperature
    average_precipitation = df_daily["precipitation_mean"].mean()
    average_temperature = df_daily["air_temperature_mean"].mean()
    
    # Handle None values
    if average_precipitation is None or average_temperature is None:
        return np.nan
    
    # Check if we have valid data (remove NaN values)
    df_clean = df_daily.filter(
        pl.col("precipitation_mean").is_not_null() & 
        pl.col("air_temperature_mean").is_not_null()
    )
    
    if df_clean.height == 0:
        return np.nan

    # For initial guess, use day 90 (around April 1) for precipitation and day 270 (around Oct 1) for temperature
    initial_phase_shift_guess_prec = 90
    initial_phase_shift_guess_temp = 270

    # Convert to numpy for curve_fit
    day_of_year_np = df_clean["day_of_year"].to_numpy()
    precipitation_np = df_clean["precipitation_mean"].to_numpy()
    temperature_np = df_clean["air_temperature_mean"].to_numpy()

    # Fit a sine curve to the precipitation and temperature data
    try:
        optimized_parameters_prec, _ = curve_fit(
            sine_curve, 
            day_of_year_np, 
            precipitation_np, 
            p0=[average_precipitation, 0.4, initial_phase_shift_guess_prec],
            maxfev=10000
        )
        optimized_parameters_temp, _ = curve_fit(
            sine_curve, 
            day_of_year_np, 
            temperature_np, 
            p0=[average_temperature, 5, initial_phase_shift_guess_temp],
            maxfev=10000
        )

        # The phase shifts are optimized_parameters[2]
        precipitation_seasonality = optimized_parameters_prec[2]
        temperature_seasonality = optimized_parameters_temp[2]

        # The amplitudes are optimized_parameters[1]
        amplitude_prec = optimized_parameters_prec[1]
        amplitude_temp = optimized_parameters_temp[1]

        # Calculate p_seasonality
        p_seasonality = amplitude_prec * np.sign(amplitude_temp) * np.cos(
            2 * np.pi * (precipitation_seasonality - temperature_seasonality) / 365.25
        )

        return round(p_seasonality, 2)
    except Exception:
        return np.nan


def calculate_frac_snow(df: pl.DataFrame) -> float:
    """
    Calculate fraction of precipitation falling as snow (for hours colder than 0°C)
    """
    if df.height == 0:
        return np.nan
    
    # fraction of precipitation falling as snow (for hours colder than 0°C)
    sum_precip_snow = df.filter(pl.col("air_temperature_mean") < 0)["precipitation_mean"].sum()
    sum_precip_water = df.filter(pl.col("air_temperature_mean") >= 0)["precipitation_mean"].sum()
    
    # Handle None and zero cases
    if sum_precip_snow is None:
        sum_precip_snow = 0
    if sum_precip_water is None:
        sum_precip_water = 0
    
    total_precip = sum_precip_snow + sum_precip_water
    
    if total_precip > 0:
        return round(sum_precip_snow / total_precip, 2)
    else:
        return np.nan


def calculate_high_prec_freq(df: pl.DataFrame) -> float:
    """
    Calculate frequency of high precipitation hours (≥ 5 times mean hourly precipitation) [hours/yr]
    """
    if df.height == 0:
        return np.nan
    
    # mean precipitation
    p_mean = df["precipitation_mean"].mean()
    
    if p_mean is None:
        return np.nan

    # number of hours >= 5 times mean precipitation
    n_high_prec_hours = df.filter(pl.col("precipitation_mean") >= 5 * p_mean).height
    
    # calculate time span in hours
    time_span_hours = (df["time"].max() - df["time"].min()).total_seconds() / 3600
    
    # frequency per year
    n_hours_high_freq = n_high_prec_hours / time_span_hours * 8760  # 8760 hours in a year

    return round(n_hours_high_freq, 2)


def calculate_high_prec_dur(df: pl.DataFrame) -> float:
    """
    Calculate average duration of high precipitation events 
    (number of consecutive hours ≥ 5 times mean hourly precipitation) [hours]
    """
    if df.height == 0:
        return np.nan
    
    # mean precipitation
    p_mean = df["precipitation_mean"].mean()
    
    if p_mean is None:
        return np.nan

    # initialize variables to keep track of high precipitation events
    high_precip_streaks = []
    current_streak = 0

    # iterate over the precipitation values
    for precip in df["precipitation_mean"].to_list():
        if precip is not None and precip >= 5 * p_mean:
            current_streak += 1
        elif current_streak > 0:
            high_precip_streaks.append(current_streak)
            current_streak = 0

    # if there's a current streak at the end of the DataFrame, add it to the list
    if current_streak > 0:
        high_precip_streaks.append(current_streak)

    # calculate the average streak length for the station
    average_streak_length = sum(high_precip_streaks) / len(high_precip_streaks) if high_precip_streaks else 0

    return round(average_streak_length, 2) if average_streak_length > 0 else np.nan


def calculate_high_prec_timing(df: pl.DataFrame) -> str:
    """
    Calculate season during which most high precipitation hours (≥ 5 times mean hourly precipitation) occur
    [season: djf, mam, jja, son]
    """
    def get_season(month):
        if month in [12, 1, 2]:
            return 'djf'
        elif month in [3, 4, 5]:
            return 'mam'
        elif month in [6, 7, 8]:
            return 'jja'
        else:
            return 'son'
    
    if df.height == 0:
        return np.nan

    # mean precipitation
    p_mean = df["precipitation_mean"].mean()
    
    if p_mean is None:
        return np.nan

    # Add month column
    df = df.with_columns([
        pl.col("time").dt.month().alias("month")
    ])
    
    # Filter high precipitation hours
    df_high_prec = df.filter(pl.col("precipitation_mean") >= 5 * p_mean)
    
    # Initialize a dictionary to keep track of the number of high-precipitation hours in each season
    season_counts = {'djf': 0, 'mam': 0, 'jja': 0, 'son': 0}

    # Count high precipitation hours per season
    for month in df_high_prec["month"].to_list():
        if month is not None:
            season_counts[get_season(month)] += 1

    # Find the season with the most high-precipitation hours
    max_count = max(season_counts.values())
    max_seasons = [season for season, count in season_counts.items() if count == max_count]

    # If there's a tie, return NaN
    if len(max_seasons) > 1 or max_count == 0:
        return np.nan
    else:
        return max_seasons[0]


def calculate_low_prec_freq(df: pl.DataFrame) -> float:
    """
    Calculate frequency of low precipitation hours (< 1 mm/hour) [hours/yr]
    """
    if df.height == 0:
        return np.nan

    # number of hours < 1 mm of precipitation
    n_low_prec_hours = df.filter(pl.col("precipitation_mean") < 1).height
    
    # calculate time span in hours
    time_span_hours = (df["time"].max() - df["time"].min()).total_seconds() / 3600
    
    # frequency per year
    n_hours_low_freq = n_low_prec_hours / time_span_hours * 8760  # 8760 hours in a year

    return round(n_hours_low_freq, 2)


def calculate_low_prec_dur(df: pl.DataFrame) -> float:
    """
    Calculate average duration of dry periods (number of consecutive hours < 1 mm/hour) [hours]
    """
    if df.height == 0:
        return np.nan

    # initialize variables to keep track of low precipitation events
    low_precip_streaks = []
    current_streak = 0

    # iterate over the precipitation values
    for precip in df["precipitation_mean"].to_list():
        if precip is not None and precip < 1:
            current_streak += 1
        elif current_streak > 0:
            low_precip_streaks.append(current_streak)
            current_streak = 0

    # if there's a current streak at the end of the DataFrame, add it to the list
    if current_streak > 0:
        low_precip_streaks.append(current_streak)

    # calculate the average streak length for the station
    average_streak_length = sum(low_precip_streaks) / len(low_precip_streaks) if low_precip_streaks else 0

    return round(average_streak_length, 2) if average_streak_length > 0 else np.nan


def calculate_low_prec_timing(df: pl.DataFrame) -> str:
    """
    Calculate season during which most dry hours (< 1mm/hour) occur
    [season: djf, mam, jja, son]
    """
    def get_season(month):
        if month in [12, 1, 2]:
            return 'djf'
        elif month in [3, 4, 5]:
            return 'mam'
        elif month in [6, 7, 8]:
            return 'jja'
        else:
            return 'son'
    
    if df.height == 0:
        return np.nan

    # Add month column
    df = df.with_columns([
        pl.col("time").dt.month().alias("month")
    ])
    
    # Filter low precipitation hours
    df_low_prec = df.filter(pl.col("precipitation_mean") < 1)
    
    # Initialize a dictionary to keep track of the number of low-precipitation hours in each season
    season_counts = {'djf': 0, 'mam': 0, 'jja': 0, 'son': 0}

    # Count low precipitation hours per season
    for month in df_low_prec["month"].to_list():
        if month is not None:
            season_counts[get_season(month)] += 1

    # Find the season with the most low-precipitation hours
    max_count = max(season_counts.values())
    max_seasons = [season for season, count in season_counts.items() if count == max_count]

    # If there's a tie, return NaN
    if len(max_seasons) > 1 or max_count == 0:
        return np.nan
    else:
        return max_seasons[0]

## Main Processing Loop

Loop through all station IDs once, reading data once per station and calculating all attributes.

**Important:**
- Work directly with hourly data (metrics are in hours, not days)
- p_seasonality internally aggregates to daily for seasonal pattern analysis

In [11]:
import polars as pl

# Reset results dataframe
df_results = pd.DataFrame()

# Main processing loop - read data once per station
for id in tqdm(ids):
    try:
        # Read camels de hydromet timeseries data
        df = pl.read_csv(
            f"../CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_{id}.csv",
            try_parse_dates=True
        )

        # Remove timezone from time column BEFORE other operations
        if df["time"].dtype.time_zone is not None:
            df = df.with_columns([
                pl.col("time").dt.replace_time_zone(None).alias("time")
            ])

        # Make all columns except 'time' float64
        df = df.with_columns([
            pl.col(col).cast(pl.Float64) for col in df.columns if col != "time"
        ])
        
        # Filter complete hydrological years
        df = filter_complete_hydro_years(df)

        if df.height == 0:
            print(f"No complete hydrological years for station {id}. Skipping.")
            # Set all attributes to NaN
            df_results.loc[id, "p_mean"] = np.nan
            df_results.loc[id, "p_seasonality"] = np.nan
            df_results.loc[id, "frac_snow"] = np.nan
            df_results.loc[id, "precipitation_perc_complete"] = np.nan
            df_results.loc[id, "high_prec_freq"] = np.nan
            df_results.loc[id, "high_prec_dur"] = np.nan
            df_results.loc[id, "high_prec_timing"] = np.nan
            df_results.loc[id, "low_prec_freq"] = np.nan
            df_results.loc[id, "low_prec_dur"] = np.nan
            df_results.loc[id, "low_prec_timing"] = np.nan
            continue

        # Calculate all attributes using the functions (working directly with hourly data)
        df_results.loc[id, "p_mean"] = calculate_p_mean(df)
        df_results.loc[id, "p_seasonality"] = calculate_p_seasonality(df)
        df_results.loc[id, "frac_snow"] = calculate_frac_snow(df)
        df_results.loc[id, "precipitation_perc_complete"] = calculate_precipitation_perc_complete(df)
        df_results.loc[id, "high_prec_freq"] = calculate_high_prec_freq(df)
        df_results.loc[id, "high_prec_dur"] = calculate_high_prec_dur(df)
        df_results.loc[id, "high_prec_timing"] = calculate_high_prec_timing(df)
        df_results.loc[id, "low_prec_freq"] = calculate_low_prec_freq(df)
        df_results.loc[id, "low_prec_dur"] = calculate_low_prec_dur(df)
        df_results.loc[id, "low_prec_timing"] = calculate_low_prec_timing(df)
        
    except Exception as e:
        print(f"Error processing station {id}: {e}")
        # Set all attributes to NaN on error
        df_results.loc[id, "p_mean"] = np.nan
        df_results.loc[id, "p_seasonality"] = np.nan
        df_results.loc[id, "frac_snow"] = np.nan
        df_results.loc[id, "precipitation_perc_complete"] = np.nan
        df_results.loc[id, "high_prec_freq"] = np.nan
        df_results.loc[id, "high_prec_dur"] = np.nan
        df_results.loc[id, "high_prec_timing"] = np.nan
        df_results.loc[id, "low_prec_freq"] = np.nan
        df_results.loc[id, "low_prec_dur"] = np.nan
        df_results.loc[id, "low_prec_timing"] = np.nan

# Display results
df_results.head()

100%|██████████| 1611/1611 [06:26<00:00,  4.16it/s]


,p_mean,p_seasonality,frac_snow,precipitation_perc_complete,high_prec_freq,high_prec_dur,high_prec_timing,low_prec_freq,low_prec_dur,low_prec_timing
DEA10180,0.11,-0.05,0.03,99.96,495.48,2.18,djf,8433.36,49.83,mam
DE710380,0.09,0.10,0.04,100.00,462.45,2.17,djf,8515.43,62.98,mam
DE212460,0.09,0.36,0.05,99.06,461.11,2.50,jja,8447.74,70.65,mam
DEE10540,0.08,-0.01,0.09,99.94,463.77,2.27,djf,8531.37,70.56,mam
DEA10570,0.09,0.05,0.03,99.95,473.99,2.12,djf,8510.45,62.01,mam


In [12]:
# save results to csv
df_results.to_csv("../CAMELS-DE-1h/CAMELS_DE_1h_climatic_attributes.csv", index_label="gauge_id")